# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — connect DuckDB to local warehouse parquet files

The warehouse was downloaded from Hugging Face using `snapshot_download` and cached locally.
DuckDB reads the local parquet files directly — no network needed after the initial download.

In [1]:
import os, glob
import duckdb
import pandas as pd
import numpy as np

# Path to the locally cached warehouse snapshot
SNAPSHOT = os.path.join(
    os.path.expanduser('~'),
    '.cache', 'huggingface', 'hub',
    'datasets--FlyRank--internship-warehouse',
    'snapshots', '50cbf7c3909d07be4d1b5906b4d09e882e5acbf2'
)

con = duckdb.connect()

# Table paths (local parquet)
TABLES = {
    'dim_clients':  f"read_parquet('{SNAPSHOT}/dim_clients.parquet')",
    'dim_content':  f"read_parquet('{SNAPSHOT}/dim_content.parquet')",
    'fact_march':   f"read_parquet('{SNAPSHOT}/fact_content_daily_performance/month=2026-03/*.parquet')",
    'fact_april':   f"read_parquet('{SNAPSHOT}/fact_content_daily_performance/month=2026-04/*.parquet')",
    'fact_sample':  f"read_parquet('{SNAPSHOT}/fact_content_daily_performance_sample.parquet')",
}

# Smoke test: confirm all files are readable
for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:16} {n:>12,} rows')

print('\nLocal parquet connection OK.')

dim_clients               104 rows
dim_content           519,606 rows
fact_march          9,841,378 rows
fact_april         10,424,730 rows
fact_sample        11,694,072 rows

Local parquet connection OK.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### The contract in five plain-words answers

1. **One row = one content page** (identified by `content_hash_id`), with daily performance aggregated over a single calendar month.

2. **Table(s):** `fact_content_daily_performance` (partitioned by month, ~79M rows total, grain = `report_date × client_hash_id × content_hash_id`), joined with `dim_content` for content metadata and `dim_clients` for per-client history coverage.

3. **Time window:** Feature window = **March 2026** (`month=2026-03`). Label window = **April 2026** (`month=2026-04`). Features are computed from the feature window; the label is computed by comparing April impressions to March impressions. This separation means features are always known *before* the outcome they predict.

4. **Target (label):** `is_declining` — 1 if total impressions in April dropped by more than 20% compared to March (i.e., `imp_april < 0.8 × imp_march`), 0 otherwise. This is an **observed outcome** (real impression counts from Google Search Console), binarized at a defined threshold (the -20% cutoff is a business choice, matching the starter's definition).

5. **Deliberately excluded:**
   - `trend_direction`, `trend_pct` — label-derived columns from the starter CSV, never features
   - All `*_last_30d` / `*_prev_30d` columns — these directly determine the starter's label
   - `provider_used`, `model_used` — product metadata, not search performance signals
   - `content_hash_id`, `client_hash_id` — for grouping/splitting only, never features
   - Any April (label-window) data used as a feature — that is leakage by definition

In [2]:
# Show the iteration months
FEATURE_MONTH = '2026-03'
LABEL_MONTH   = '2026-04'

print(f'Feature window: {FEATURE_MONTH} (build features here)')
print(f'Label window:   {LABEL_MONTH} (compare to feature window to define decline)')
print()
print('One row in the feature frame = one content page,')
print('aggregated from daily facts over March 2026.')
print()
print('Label logic:')
print('  is_declining = 1  if  SUM(impressions in April) < 0.8 * SUM(impressions in March)')
print('               = 0  otherwise')
print()
print('Why 0.8x? A >20% drop matches the starter definition of "down".')
print('The key improvement: features (March) are strictly BEFORE the label (April).')

Feature window: 2026-03 (build features here)
Label window:   2026-04 (compare to feature window to define decline)

One row in the feature frame = one content page,
aggregated from daily facts over March 2026.

Label logic:
  is_declining = 1  if  SUM(impressions in April) < 0.8 * SUM(impressions in March)
               = 0  otherwise

Why 0.8x? A >20% drop matches the starter definition of "down".
The key improvement: features (March) are strictly BEFORE the label (April).


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Fields | Notes |
|--------|--------|-------|
| **Feature** | `total_impressions`, `total_clicks`, `avg_position`, `days_active`, `ctr` | All computed from the feature-window (March) daily facts. Knowable before the label window. |
| **Label** | `is_declining` | 1 if April impressions < 0.8 × March impressions. Observed outcome. |
| **Context** | `content_hash_id`, `client_hash_id` | For grouping, joining, client-holdout splits. Never model features. |
| **Excluded** | `trend_direction`, `trend_pct` | Label-derived in the starter CSV. |
| **Excluded** | `provider_used`, `model_used` | Product metadata, not search performance signals. |
| **Excluded** | April daily facts used as features | Future information = leakage. |

In [3]:
# Show the field classification programmatically
field_classification = pd.DataFrame([
    ('total_impressions',  'Feature',  'SUM(gsc_impressions) over March',     'Knowable: aggregated from feature window only'),
    ('total_clicks',       'Feature',  'SUM(gsc_clicks) over March',          'Knowable: aggregated from feature window only'),
    ('avg_position',       'Feature',  'AVG(gsc_avg_position) over March',    'Knowable: average position in feature window'),
    ('days_active',        'Feature',  'COUNT(days with impressions>0)',       'Knowable: counted in feature window only'),
    ('ctr',                'Feature',  'total_clicks / total_impressions',     'Knowable: derived from feature-window aggregates'),
    ('is_declining',       'Label',    'April imps < 0.8 * March imps',       'Observed: real impression counts, defined threshold'),
    ('content_hash_id',    'Context',  'Page identifier',                     'Grouping/joins only, never a feature'),
    ('client_hash_id',     'Context',  'Client identifier',                   'Client-holdout splits, never a feature'),
    ('trend_direction',    'Excluded', 'Label-derived',                        'IS the label in the starter CSV'),
    ('trend_pct',          'Excluded', 'Label-derived',                        'Continuous version of the label'),
    ('provider_used',      'Excluded', 'Product metadata',                     'Not a search performance signal'),
    ('April daily facts',  'Excluded', 'Future information',                   'Label window data = leakage as features'),
], columns=['Field', 'Bucket', 'Definition', 'Rationale'])

print(field_classification.to_string(index=False))

            Field   Bucket                       Definition                                           Rationale
total_impressions  Feature  SUM(gsc_impressions) over March       Knowable: aggregated from feature window only
     total_clicks  Feature       SUM(gsc_clicks) over March       Knowable: aggregated from feature window only
     avg_position  Feature AVG(gsc_avg_position) over March        Knowable: average position in feature window
      days_active  Feature   COUNT(days with impressions>0)            Knowable: counted in feature window only
              ctr  Feature total_clicks / total_impressions    Knowable: derived from feature-window aggregates
     is_declining    Label    April imps < 0.8 * March imps Observed: real impression counts, defined threshold
  content_hash_id  Context                  Page identifier                Grouping/joins only, never a feature
   client_hash_id  Context                Client identifier              Client-holdout splits, never a 

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1: Grain check
The daily fact table's grain is `report_date × client_hash_id × content_hash_id`. Verify: no duplicate (date, client, content) combos in March.

In [4]:
# QUERY 1: Grain check on the daily fact for March 2026
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {TABLES['fact_march']}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING n > 1
    LIMIT 5
""").df()

if len(grain_check) == 0:
    print('GRAIN CHECK PASSED: zero duplicate (date, client, content) rows in March 2026.')
    print('One row = one content item per day, as expected.')
else:
    print('GRAIN CHECK FAILED -- duplicates found:')
    print(grain_check)

GRAIN CHECK PASSED: zero duplicate (date, client, content) rows in March 2026.
One row = one content item per day, as expected.


### Query 2: Row count and date span for the feature month

In [5]:
# QUERY 2: Row count and date span for March 2026
counts = con.sql(f"""
    SELECT
        COUNT(*)                          AS total_rows,
        COUNT(DISTINCT content_hash_id)   AS distinct_content,
        COUNT(DISTINCT client_hash_id)    AS distinct_clients,
        MIN(report_date)                  AS min_date,
        MAX(report_date)                  AS max_date,
        COUNT(DISTINCT report_date)       AS distinct_days
    FROM {TABLES['fact_march']}
""").df()

print('March 2026 partition summary:')
print(counts.to_string(index=False))
print()
print(f'Dates span {counts["min_date"].iloc[0]} to {counts["max_date"].iloc[0]}')
print(f'{counts["distinct_content"].iloc[0]:,} unique content items across {counts["distinct_clients"].iloc[0]} clients')
print(f'{counts["total_rows"].iloc[0]:,} daily rows over {counts["distinct_days"].iloc[0]} days')

March 2026 partition summary:
 total_rows  distinct_content  distinct_clients   min_date   max_date  distinct_days
    9841378            331437                55 2026-03-01 2026-03-31             31

Dates span 2026-03-01 00:00:00 to 2026-03-31 00:00:00
331,437 unique content items across 55 clients
9,841,378 daily rows over 31 days


### Query 3: Availability — filter with `ga4_data_available IS TRUE`

In [6]:
# QUERY 3: Availability -- how many rows survive ga4_data_available IS TRUE?
avail = con.sql(f"""
    SELECT
        COUNT(*)                                                                    AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)                          AS ga4_available_rows,
        ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)
              / COUNT(*), 1)                                                        AS pct_ga4_available,
        COUNT(DISTINCT client_hash_id)                                              AS total_clients,
        COUNT(DISTINCT client_hash_id) FILTER (WHERE ga4_data_available IS TRUE)    AS clients_with_ga4
    FROM {TABLES['fact_march']}
""").df()

print('GA4 availability in March 2026:')
print(avail.to_string(index=False))
print()
pct = avail['pct_ga4_available'].iloc[0]
print(f'{pct}% of rows have GA4 data available.')
print('Rows without GA4 have engagement columns zero-filled --')
print('filter on the flag, do not treat those zeros as "no engagement".')

GA4 availability in March 2026:
 total_rows  ga4_available_rows  pct_ga4_available  total_clients  clients_with_ga4
    9841378              413966                4.2             55                41

4.2% of rows have GA4 data available.
Rows without GA4 have engagement columns zero-filled --
filter on the flag, do not treat those zeros as "no engagement".


### Five features — built from the feature window (March 2026)

Each feature gets one line: *"knowable at the decision moment because..."*

The label comes from April — a separate month the features never see.

In [7]:
# Build a five-feature frame from March 2026, with the label from April 2026
features_df = con.sql(f"""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions)                                  AS total_impressions,
            SUM(gsc_clicks)                                       AS total_clicks,
            AVG(CASE WHEN gsc_avg_position > 0
                     THEN gsc_avg_position END)                   AS avg_position,
            COUNT(*) FILTER (WHERE gsc_impressions > 0)           AS days_active
        FROM {TABLES['fact_march']}
        GROUP BY client_hash_id, content_hash_id
        HAVING total_impressions >= 10  -- minimum volume filter
    ),
    april AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS imp_april
        FROM {TABLES['fact_april']}
        GROUP BY content_hash_id
    )
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        -- FEATURES (all from March = feature window)
        m.total_impressions,
        m.total_clicks,
        m.avg_position,
        m.days_active,
        CASE WHEN m.total_impressions > 0
             THEN ROUND(100.0 * m.total_clicks / m.total_impressions, 2)
             ELSE 0 END                                          AS ctr,
        -- LABEL (from April = label window)
        COALESCE(a.imp_april, 0)                                  AS imp_april,
        CASE WHEN COALESCE(a.imp_april, 0) < 0.8 * m.total_impressions
             THEN 1 ELSE 0 END                                   AS is_declining
    FROM march m
    LEFT JOIN april a ON m.content_hash_id = a.content_hash_id
""").df()

print(f'Feature frame: {len(features_df):,} rows (one per content page with >= 10 March impressions)')
print(f'Declining: {features_df["is_declining"].sum():,} ({features_df["is_declining"].mean()*100:.1f}%)')
print(f'Not declining: {(1-features_df["is_declining"]).sum():,.0f} ({(1-features_df["is_declining"]).mean()*100:.1f}%)')
print()
print('Feature descriptions ("knowable at decision moment because..."):')
print('  1. total_impressions -- SUM of daily GSC impressions in March.')
print('     Knowable: it is the feature window itself.')
print('  2. total_clicks      -- SUM of daily GSC clicks in March.')
print('     Knowable: same window as impressions.')
print('  3. avg_position      -- AVG daily GSC position in March (excluding 0 = no data).')
print('     Knowable: averaged within the feature window.')
print('  4. days_active       -- days in March with at least 1 impression.')
print('     Knowable: counted in the feature window only.')
print('  5. ctr               -- total_clicks / total_impressions * 100.')
print('     Knowable: derived from feature-window aggregates.')
print()
features_df.head(10)

Feature frame: 143,206 rows (one per content page with >= 10 March impressions)
Declining: 74,005 (51.7%)
Not declining: 69,201 (48.3%)

Feature descriptions ("knowable at decision moment because..."):
  1. total_impressions -- SUM of daily GSC impressions in March.
     Knowable: it is the feature window itself.
  2. total_clicks      -- SUM of daily GSC clicks in March.
     Knowable: same window as impressions.
  3. avg_position      -- AVG daily GSC position in March (excluding 0 = no data).
     Knowable: averaged within the feature window.
  4. days_active       -- days in March with at least 1 impression.
     Knowable: counted in the feature window only.
  5. ctr               -- total_clicks / total_impressions * 100.
     Knowable: derived from feature-window aggregates.



,client_hash_id,content_hash_id,total_impressions,total_clicks,avg_position,days_active,ctr,imp_april,is_declining
0,client_1a730cb2640a1abf,content_0ecd5a0bd7013196,26.0,0.0,14.366667,8,0.00,36.0,0
1,client_0fa64a184f18a4a0,content_62dae8bf28a640ec,21.0,0.0,8.119048,14,0.00,14.0,1
2,client_0fa64a184f18a4a0,content_eed6e35dedad8bb0,10.0,0.0,7.104167,5,0.00,4.0,1
3,client_0fa64a184f18a4a0,content_72e894f4da877975,94.0,1.0,3.038511,21,1.06,95.0,0
4,client_0fa64a184f18a4a0,content_b04fd864d1ab218c,370.0,0.0,2.010124,19,0.00,11.0,1
5,client_0fa64a184f18a4a0,content_305c1fa4cc05b8b2,199.0,1.0,2.586484,19,0.50,547.0,0
6,client_0fa64a184f18a4a0,content_1dfa69b3a2e5c9ed,25.0,1.0,4.977778,9,4.00,420.0,0
7,client_0fa64a184f18a4a0,content_9d8fadf61f1d0589,49.0,0.0,4.656158,5,0.00,0.0,1
8,client_ccdd78843409c8c7,content_3949f5f3f36390a4,125.0,0.0,8.384894,30,0.00,0.0,1
9,client_b77d0d5f08f05e64,content_684c162bda86abf8,12.0,0.0,29.516667,4,0.00,3.0,1


### The trap: deliberate leakage experiment

The assignment says: *add ONE label-derived column on purpose, watch your quick score jump toward perfect, then delete it and keep the honest number.*

`imp_april` (the label-period impressions) is already in the dataframe. Since `is_declining` is literally computed from `imp_april`, including it as a feature is textbook leakage — the model reads the answer from the input.

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Prepare data (drop rows with missing avg_position for clean comparison)
model_df = features_df.dropna(subset=['avg_position']).copy()

honest_features = ['total_impressions', 'total_clicks', 'avg_position', 'days_active', 'ctr']
leaked_features = honest_features + ['imp_april']  # <-- LEAKAGE: label-period data

y = model_df['is_declining']

# --- HONEST model (no leakage) ---
X_honest = model_df[honest_features]
X_tr, X_te, y_tr, y_te = train_test_split(
    X_honest, y, test_size=0.25, random_state=42, stratify=y)
rf_honest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_honest.fit(X_tr, y_tr)
auc_honest = roc_auc_score(y_te, rf_honest.predict_proba(X_te)[:, 1])

# --- LEAKED model (with label-period impressions as feature) ---
X_leaked = model_df[leaked_features]
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(
    X_leaked, y, test_size=0.25, random_state=42, stratify=y)
rf_leaked = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_leaked.fit(X_tr_l, y_tr_l)
auc_leaked = roc_auc_score(y_te_l, rf_leaked.predict_proba(X_te_l)[:, 1])

print('LEAKAGE EXPERIMENT')
print('=' * 60)
print(f'Honest model (5 features, no leakage):  AUC = {auc_honest:.3f}')
print(f'Leaked model (+imp_april as feature):   AUC = {auc_leaked:.3f}')
print()
print(f'The leaked AUC jumped by {auc_leaked - auc_honest:+.3f}.')
print('This is because imp_april IS the label -- the model reads')
print('the answer directly from the input.')
print()
print('VERDICT: imp_april is DELETED from the feature set.')
print(f'Honest AUC = {auc_honest:.3f} is the number we keep.')

LEAKAGE EXPERIMENT
Honest model (5 features, no leakage):  AUC = 0.607
Leaked model (+imp_april as feature):   AUC = 0.999

The leaked AUC jumped by +0.393.
This is because imp_april IS the label -- the model reads
the answer directly from the input.

VERDICT: imp_april is DELETED from the feature set.
Honest AUC = 0.607 is the number we keep.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitations of this slice:**

1. **Unbalanced panel -- not all clients have March 2026 data.** History depth differs per client. Some clients started tracking in mid-2025; others only in 2026. A global March-April window may exclude clients entirely or include clients with very short history. Always check `dim_clients.gsc_data_start`.

2. **GSC-only early rows.** Rows before a client's `ga4_data_start` have GA4 engagement columns zero-filled with `ga4_data_available = FALSE`. My current five features use only GSC signals, so this doesn't bite yet -- but any future engagement features (sessions, scroll rate) must filter by the flag first.

3. **The label is threshold-defined.** `is_declining` uses a -20% cutoff. The model learns this threshold's pattern, not some deeper truth about "real" decline. A different threshold yields a different model.

4. **Seasonality, consolidation, and noise are not controlled for.** A page's impressions may drop in April for seasonal reasons (not true decline), or because a sibling page absorbed its traffic (consolidation). This label does not distinguish these from genuine decay.

5. **No causal claims.** This is observational data. We can say "pages with feature X tend to decline more" -- never "feature X causes decline."

**The output this analysis hands to a human:** A ranked list of content pages scored by predicted decline probability. An editor reviews the top of the list and schedules content refreshes on the highest-risk pages first. This is decision-support, not an automated action.

In [9]:
# Show the per-client history coverage for March 2026
client_coverage = con.sql(f"""
    SELECT
        dc.client_hash_id,
        dc.gsc_data_start,
        dc.ga4_data_start,
        COUNT(DISTINCT f.report_date) AS days_in_march,
        COUNT(DISTINCT f.content_hash_id) AS content_items
    FROM {TABLES['dim_clients']} dc
    LEFT JOIN {TABLES['fact_march']} f
        ON dc.client_hash_id = f.client_hash_id
    GROUP BY dc.client_hash_id, dc.gsc_data_start, dc.ga4_data_start
    ORDER BY days_in_march DESC
""").df()

has_march = (client_coverage['days_in_march'] > 0).sum()
total = len(client_coverage)
print(f'Client coverage for March 2026: {has_march} of {total} clients have data')
print(f'{total - has_march} clients have NO daily data in March -- unbalanced panel.')
print()
print('Top 10 clients by coverage:')
print(client_coverage.head(10).to_string(index=False))

Client coverage for March 2026: 55 of 104 clients have data
49 clients have NO daily data in March -- unbalanced panel.

Top 10 clients by coverage:
         client_hash_id gsc_data_start ga4_data_start  days_in_march  content_items
client_fef1a8f436438636     2025-03-11     2026-03-06             31          11223
client_3ffa76342f366962     2025-10-11     2026-03-11             31          31108
client_b10cb2997d0c7c86     2025-06-18     2025-11-15             31           4325
client_0fa64a184f18a4a0     2026-02-19     2026-02-17             31           1766
client_b77d0d5f08f05e64     2026-03-12     2026-03-19             31           1143
client_c182d11e4862a37d     2025-06-21     2026-02-20             31           1864
client_a60a11451483af1c     2025-11-16     2026-02-19             31           5457
client_a2eeb8899886adde     2025-07-06     2026-02-19             31           1220
client_2c32078d69f2cbad     2025-11-05            NaT             31             11
client_2910

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled -- markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Five plain-words contract answers in section 1
- [x] Three verification queries with visible outputs (grain, counts, availability with IS TRUE)
- [x] Five-feature frame with "available when?" line per feature
- [x] Deliberate-leak experiment shown and removed
- [x] One named limitation of the slice
- [ ] Committed to my repo under `work/notebooks/` -- then submit your repo URL on the card. Done.